# Plan de modèle prédictif — Délinquance Geldium (Tata iQ, partie 2)

Ce notebook conceptualise un modèle de prédiction du risque de défaut de paiement à partir des enseignements de l'EDA (partie 1) :

1. Générer la logique du modèle avec la GenAI
2. Justifier le choix du modèle
3. Planifier l'évaluation de la performance et de l'équité
4. Produire le plan (`Model_Plan_Geldium.docx`)

Le code est illustratif : l'objectif est de montrer le raisonnement et le flux de travail, pas de livrer un modèle de production.

---
## Étape 1 — Générer la logique du modèle avec la GenAI

### Prompts GenAI utilisés
- « Esquisse un pipeline de modélisation prédictive pour prévoir les retards de crédit, de la sélection des variables à l'évaluation du modèle. »
- « Propose deux options de modélisation (une simple et une complexe) pour prédire la délinquance, et recommandes-en une. »
- « Explique comment le modèle transforme les variables d'entrée d'un client en une prédiction finale du risque, de l'ingestion des données à la sortie. »
- « Génère un pseudocode clair pour construire un modèle de risque de crédit avec le revenu, l'utilisation du crédit et les paiements manqués. »

### 1.1 Deux options de modélisation

| | Option simple : **régression logistique** | Option complexe : **gradient boosting** (XGBoost / LightGBM) |
|---|---|---|
| Principe | Combinaison linéaire pondérée des variables, transformée en probabilité par la fonction sigmoïde | Ensemble de centaines d'arbres de décision construits séquentiellement, chacun corrigeant les erreurs du précédent |
| Interprétabilité | Élevée : chaque coefficient se lit comme un effet sur le risque (odds ratio) | Faible en natif, nécessite SHAP pour expliquer les décisions |
| Interactions non linéaires | À créer manuellement | Capturées automatiquement |
| Volume de données requis | Fonctionne avec 500 lignes | Risque élevé de surapprentissage sur 500 lignes |
| Usage dans le secteur | Standard des scorecards de crédit réglementées | Utilisé comme modèle challenger ou en détection de fraude |

**Recommandation : la régression logistique** comme modèle principal, avec le gradient boosting comme *challenger* pour vérifier qu'aucune relation non linéaire importante n'est ignorée.

### 1.2 Flux de travail : de l'ingestion à la prédiction

```
Données client (Excel / base Geldium)
   │
   ▼
1. Ingestion et validation ── contrôle du schéma, des plages (Utilisation 0–1,2 ; Score 300–850), de la cohérence âge/ancienneté
   │
   ▼
2. Nettoyage ── normalisation d'Employment_Status, imputation médiane + indicateurs de valeur manquante (cf. EDA)
   │
   ▼
3. Ingénierie des variables ── Missed_6m, Late_6m, Over_Limit, Recent_Missed (Month_6), New_Account (Tenure = 0)
   │
   ▼
4. Prétraitement ── standardisation des numériques, one-hot des catégorielles ; exclusion de Location et Age (équité)
   │
   ▼
5. Modèle ── régression logistique pondérée (class_weight = "balanced", 16 % de délinquants)
   │
   ▼
6. Sortie ── probabilité de défaut (0–1) → bande de risque par rang (Élevé = top 10 %, Moyen = 10–30 %, Faible = reste) → action
```

### 1.3 Pseudocode

```
CHARGER données clients
VALIDER schéma et plages ; SIGNALER les profils incohérents
NETTOYER : normaliser Employment_Status ; imputer Income (médiane par emploi), Loan_Balance, Credit_Score
CRÉER variables : Missed_6m, Late_6m, Recent_Missed, Over_Limit, New_Account
X ← [historique de paiement, Credit_Utilization, Debt_to_Income_Ratio, Credit_Score, Employment_Status, ...]
y ← Delinquent_Account
DIVISER en entraînement (80 %) / test (20 %), stratifié sur y
PIPELINE ← Standardisation(numériques) + OneHot(catégorielles) + RégressionLogistique(poids équilibrés)
ENTRAÎNER PIPELINE sur entraînement (validation croisée 5 plis pour régler C)
POUR chaque client :
    p ← PIPELINE.predict_proba(client)          # probabilité calibrée (CalibratedClassifierCV)
    bande ← "Élevé" si p dans les 10 % les plus risqués ; "Moyen" si dans les 30 % ; sinon "Faible"
    EXPLIQUER : contributions coefficient × valeur des 3 principales variables
RETOURNER p, bande, explication
```

### 1.4 Esquisse de code (illustrative)

In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

TARGET = "Delinquent_Account"
df = pd.read_csv("../Delinquency_prediction_cleaned.csv")  # jeu nettoyé à l'étape 2 de l'EDA

# Ingénierie des variables issues de l'EDA
df["Recent_Missed"] = (df.Month_6 == "Missed").astype(int)
df["New_Account"] = (df.Account_Tenure == 0).astype(int)

NUMERIC = ["Missed_Payments", "Missed_6m", "Late_6m", "Recent_Missed", "Credit_Utilization", "Over_Limit",
           "Debt_to_Income_Ratio", "Credit_Score", "Income", "Loan_Balance", "Account_Tenure", "New_Account",
           "Income_missing", "Loan_Balance_missing"]
CATEGORICAL = ["Employment_Status", "Credit_Card_Type"]
# Exclues volontairement : Location et Age (risque de discrimination), Customer_ID (identifiant)

X, y = df[NUMERIC + CATEGORICAL], df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

model = Pipeline([
    ("prep", ColumnTransformer([
        ("num", StandardScaler(), NUMERIC),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), CATEGORICAL),
    ])),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000)),
])
model.fit(X_train, y_train)
print(f"Entraînement : {len(X_train)} clients, test : {len(X_test)} clients")

Entraînement : 400 clients, test : 100 clients


### 1.5 Des variables d'entrée à la décision
Le modèle attribue une probabilité à chaque client du jeu de test. Les bandes de risque sont définies **par rang** (top 10 % = Élevé, 10–30 % = Moyen) plutôt que par un seuil fixe : la pondération des classes gonfle les probabilités absolues, qui devront être recalibrées (`CalibratedClassifierCV`) avant d'utiliser des seuils fixes. Le classement par rang permet aussi d'adapter le volume d'interventions à la capacité des équipes.

In [2]:
ACTIONS = {"Faible": "Aucune action", "Moyen": "Rappel automatique avant échéance", "Élevé": "Contact par un conseiller"}

scored = X_test.copy()
scored["proba_defaut"] = model.predict_proba(X_test)[:, 1].round(3)
rank = scored.proba_defaut.rank(pct=True, method="first")
scored["bande"] = pd.cut(rank, [0, 0.7, 0.9, 1], labels=["Faible", "Moyen", "Élevé"])
scored["action"] = scored.bande.map(ACTIONS)

print(scored.bande.value_counts().reindex(["Élevé", "Moyen", "Faible"]).to_dict())
cols = ["Missed_Payments", "Credit_Utilization", "Debt_to_Income_Ratio", "Credit_Score", "Employment_Status",
        "proba_defaut", "bande", "action"]
scored.sort_values("proba_defaut", ascending=False)[cols].iloc[[0, 1, 15, 16, 98, 99]]

{'Élevé': 10, 'Moyen': 20, 'Faible': 70}


,Missed_Payments,Credit_Utilization,Debt_to_Income_Ratio,Credit_Score,Employment_Status,proba_defaut,bande,action
52,4,0.349340,0.338566,823.0,Employed,0.814,Élevé,Contact par un conseiller
151,1,0.602569,0.229288,524.0,Unemployed,0.793,Élevé,Contact par un conseiller
432,2,0.700700,0.257964,739.0,Employed,0.610,Moyen,Rappel automatique avant échéance
224,0,0.560467,0.282422,535.0,Unemployed,0.606,Moyen,Rappel automatique avant échéance
105,6,0.626524,0.199682,340.0,Retired,0.208,Faible,Aucune action
92,6,0.396749,0.184346,372.0,Retired,0.193,Faible,Aucune action


**Point d'attention.** Les clients classés « Faible » ont ici 6 paiements manqués et un Credit_Score inférieur à 400, alors que les mieux notés figurent parmi les « Élevé ». Le modèle reproduit les relations inversées constatées dans l'EDA, ce qui confirme que **ce pipeline ne doit pas être déployé avant la fiabilisation des données** par Geldium. La logique reste valable ; ce sont les données d'entrée qui posent problème.

### 1.6 Synthèse de l'étape 1

**Type de modèle choisi.** Une **régression logistique pondérée** estime la probabilité qu'un client devienne délinquant à partir d'une combinaison linéaire de ses caractéristiques financières et de son historique de paiement. Cette probabilité est ensuite convertie en bande de risque (Faible / Moyen / Élevé) qui déclenche une action d'intervention graduée. Un modèle de gradient boosting sert de challenger pour détecter d'éventuelles relations non linéaires.

**Les 5 principales variables d'entrée**

| # | Variable | Contribution attendue à la prédiction |
|---|---|---|
| 1 | **Historique de paiement** (`Missed_Payments`, `Missed_6m`, `Recent_Missed`) | Premier prédicteur du risque de crédit : les retards passés et récents annoncent les défauts futurs. À fiabiliser (incohérence relevée dans l'EDA). |
| 2 | **Credit_Utilization** (+ `Over_Limit`) | Une utilisation élevée ou un dépassement de limite signale une tension de trésorerie. |
| 3 | **Debt_to_Income_Ratio** | Mesure la capacité de remboursement : plus le ratio est élevé, moins le client a de marge. |
| 4 | **Credit_Score** | Synthèse externe de la solvabilité, standard dans tous les modèles de crédit. |
| 5 | **Employment_Status** | La stabilité des revenus influence directement la capacité à payer (19,4 % de délinquance chez les chômeurs dans l'EDA). |

Variables secondaires : `Account_Tenure` / `New_Account` (28,6 % de délinquance pour les comptes de moins d'un an), `Credit_Card_Type`, `Income`, `Loan_Balance`.
Variables exclues : `Location` et `Age`, pour éviter une discrimination géographique ou liée à l'âge. Elles sont réservées aux contrôles d'équité (étape 3).